In [2]:
import cv2
import glob
import re
from ultralytics import YOLO
from paddleocr import PaddleOCR

model = YOLO(r"runs/detect/train-3/weights/best.pt")

ocr = PaddleOCR(use_angle_cls=True, lang='en')

images = glob.glob(r"dataset/test/images/*.jpg")
print("IMAGES:", len(images))

def clean_plate(text):
    text = text.upper()
    text = re.sub(r'[^A-Z0-9]', '', text)

    replacements = {
        "O": "0",
        "I": "1",
        "Z": "2",
        "S": "5",
        "B": "8",
        "G": "6"
    }

    for k, v in replacements.items():
        text = text.replace(k, v)

    return text


def is_valid_plate(text):
    return 6 <= len(text) <= 9

for img_path in images:

    img = cv2.imread(img_path)

    results = model.predict(img, conf=0.25, verbose=False)

    for r in results:
        for box in r.boxes:

            x1, y1, x2, y2 = map(int, box.xyxy[0])

            crop = img[y1:y2, x1:x2]

            result = ocr.predict(crop)

            text = ""

            if result and len(result) > 0:
                try:
                    text = result[0]['rec_texts'][0]
                except:
                    text = ""

            text = clean_plate(text)

            if is_valid_plate(text):
                print(img_path, "→", text)

C:\Users\lindo\AppData\Local\Temp\ipykernel_8012\2082608131.py:9: DeprecationWarning: The parameter `use_angle_cls` has been deprecated and will be removed in the future. Please use `use_textline_orientation` instead.
  ocr = PaddleOCR(use_angle_cls=True, lang='en')
Creating model: ('PP-LCNet_x1_0_doc_ori', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\lindo\.paddlex\official_models\PP-LCNet_x1_0_doc_ori`.
Creating model: ('UVDoc', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\lindo\.paddlex\official_models\UVDoc`.
Creating model: ('PP-LCNet_x1_0_textline_ori', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\lindo\.paddlex\official_models\PP-LCNet_x1_0_textline_ori`.
Creating model: ('PP-OCRv6_medium_det', None, None)
Model files already exist. Using cached files.

IMAGES: 63


NotImplementedError: (Unimplemented) ConvertPirAttribute2RuntimeAttribute not support [pir::ArrayAttribute<pir::DoubleAttribute>]  (at ..\paddle\fluid\framework\new_executor\instruction\onednn\onednn_instruction.cc:118)
